# 激活函数详解

- 📅 2026-07-18
- 🏷️ 神经网络基础, 激活函数, Softmax, Sigmoid, ReLU
- 📖 参考：《深度学习入门》、CS231n

## 1. 为什么需要激活函数

如果没有激活函数（或只使用线性函数），多层网络等价于单层：

```
不在用激活函数:  h1 = W1·x,  h2 = W2·h1
                     = W2·W1·x
                     = W'·x        ← 还是线性的!
```

**激活函数引入非线性**，让网络能拟合任意复杂函数。没有它，再深的网络也只是线性变换。

> 🎯 核心要求：激活函数必须是**非线性的**。除此之外，可微（方便反向传播）和单调（优化更稳定）是加分项。

## 2. 一图纵览五种激活函数

```
┌──────────────────────────────────────────────────────────────────────┐
│                                                                      │
│   阶跃              Sigmoid           Tanh           ReLU       Softmax│
│                                                                      │
│   ┌──┐              ┌──╮             ┌──╮           ┌──╱       ╱────│
│   │  │             ╱   ╲           ╱   ╲          ╱          ╱      │
│   │  ╰─→    ╌╌╌╌╌╌╌╌╌╌╌╌╌   ╌╌╌╌╌╌╌╌╌╌╌╌╌    ╌╌╌╌╱     ╌╌╌╌╌╌╌╌╌╌╌│
│   │  │             ╱     ╲         ╱     ╲        ╱                  │
│   └──┘           ─┘       └─     ─┘       └─    ─┘         (概率分布)│
│                                                                      │
│   感知机用        传统 NN 用       RNN 常用      现代默认      多分类输出│
│   0/1 硬切换      0→0.5             -1→1         max(0,x)    和为 1   │
│                  易饱和            零中心        不饱和        指数家族│
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (12, 4),
    'figure.dpi': 100,
    'font.size': 12
})

In [ ]:
# 五种激活函数的可视化对比

def step(x):
    return np.array(x > 0, dtype=np.float64)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

x = np.linspace(-4, 4, 500)

functions = [
    ('阶跃 Step', step(x), 'C0', ':'),
    ('Sigmoid', sigmoid(x), 'C1', '-'),
    ('Tanh', tanh(x), 'C2', '-'),
    ('ReLU', relu(x), 'C3', '-'),
    ('Leaky ReLU (α=0.01)', leaky_relu(x), 'C4', '--'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：分别画
for name, y, color, ls in functions[:4]:
    axes[0].plot(x, y, color=color, linestyle=ls, linewidth=2, label=name)
axes[0].axhline(0, color='gray', alpha=0.2)
axes[0].axvline(0, color='gray', alpha=0.2)
axes[0].set_ylim(-1.5, 4.5)
axes[0].set_title('四大激活函数')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.2)

# 右图：ReLU 家族
x2 = np.linspace(-2, 2, 300)
for alpha, color, ls in [(0, 'C3', '-'), (0.01, 'C4', '--'), (0.1, 'C1', '-.'), (1.0, 'C2', ':')]:
    axes[1].plot(x2, leaky_relu(x2, alpha), color=color, linestyle=ls, linewidth=2,
                 label=f'α={alpha}' + (' (ReLU)' if alpha == 0 else ' (Linear)' if alpha == 1 else ''))
axes[1].axhline(0, color='gray', alpha=0.2)
axes[1].axvline(0, color='gray', alpha=0.2)
axes[1].set_ylim(-0.5, 2.5)
axes[1].set_title('Leaky ReLU 家族: max(αx, x)')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 3. Sigmoid

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

### 关键性质

| 属性 | 值 |
|------|-----|
| 值域 | (0, 1) |
| 中心 | 0.5（**非零中心**，这是缺点）|
| 饱和区 | \|x\| > 3 时梯度 ≈ 0 |
| 导数 | σ'(x) = σ(x)(1-σ(x)) |
| 最大梯度 | 0.25（在 x=0 处）|

```
   σ(x)                          σ'(x)
  1 ┤         ╭─────            0.25┤    ╭─╮
    │       ╱                     │   ╱   ╲
0.5 ┤     ╱    ← 非零中心!         │  ╱     ╲
    │   ╱                        │ ╱       ╲
  0 ┤─╱──────→                0 ─┴─────────→
   -4   0    4                  -4   0    4
```

### 三大缺点

1. **梯度消失**：输入太大或太小时，导数趋近 0，深层网络传不动
2. **输出非零中心**：后续层收到全正信号，zig-zag 优化路径，收敛慢
3. **exp 计算昂贵**：相比 ReLU

## 4. Tanh

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = 2\sigma(2x) - 1$$

| 属性 | 值 |
|------|-----|
| 值域 | (-1, 1) |
| 中心 | **0（零中心 ✓）**|
| 饱和区 | \|x\| > 2 时梯度 ≈ 0 |
| 导数 | tanh'(x) = 1 - tanh²(x) |

```
   tanh(x)           vs           sigmoid
  1 ┤       ╭───                 ┌─ 零中心!
    │     ╱                        │   Tanh 输出有正有负
  0 ┤───╱────────→                │   = 2*σ(2x) - 1
    │  ╱                           │   → 只是缩放+平移
 -1 ┤╱                            └─ 仍会饱和
```

> Tanh 基本就是 Sigmoid 的零中心版本。比 Sigmoid 好，但**依然会饱和**，深层网络还是会梯度消失。现代 CNN 几乎不用。

## 5. ReLU 家族 — 现代默认选择

### 5.1 ReLU

$$\text{ReLU}(x) = \max(0, x)$$

```
  优点                              缺点
  ┌──────────────────────┐         ┌─────────────────────┐
  │ • 正区间梯度 = 1     │         │ • 输出非零中心       │
  │   → 无梯度消失       │         │ • 负区间梯度 = 0     │
  │ • 计算极快（就是比较）│         │   → Dead ReLU!      │
  │ • 自然稀疏激活        │         │   某些神经元永远不激活│
  │ • 生物合理性          │         │ • 输出无上限        │
  └──────────────────────┘         └─────────────────────┘
```

**Dead ReLU 现象**：若一个神经元的学习率过大或梯度太猛，权重更新后对所有输入都输出 0，该神经元就永远"死"了。

### 5.2 ReLU 变体——本质是给负区间"一口仙气"

| 变体 | 公式 | 负区间行为 |
|------|------|-----------|
| ReLU | max(0, x) | 完全归零 |
| Leaky ReLU | max(αx, x), α≈0.01 | 留一点点 |
| PReLU | max(αx, x), α 可学习 | 负区间斜率自己学 |
| ELU | x>0: x; x≤0: α(eˣ-1) | 负值有平滑过渡 |
| GELU | x·Φ(x)（正态分布 CDF） | 概率性"关" |
| Swish/SiLU | x·σ(x) | 平滑、处处可微 |

```
  负区间策略进化:

  ReLU:     负值 → 0         （硬杀）
  Leaky:    负值 → 0.01·x   （留条后路）
  PReLU:    负值 → α·x      （自己学怎么杀）
  ELU:      负值 → α(eˣ-1)  （平滑过渡、零中心）
  GELU:     负值 → 概率性    （Transformer 最爱）
```

## 6. Softmax — 多分类输出层

Softmax 在神经网络中有特殊地位：它**专门用于多分类问题的输出层**，把任意实数向量变成概率分布。

### 6.1 定义

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum\limits_{k=1}^{K} e^{z_k}}, \quad i = 1, 2, \dots, K$$

```
    输入 z (logits)              输出 p (概率分布)
    ┌───────────┐               ┌───────────┐
    │ z₁ = 2.0  │  ──────→     │ p₁ = 0.66 │  ← max
    │ z₂ = 1.0  │   Softmax    │ p₂ = 0.24 │
    │ z₃ = 0.1  │  ──────→     │ p₃ = 0.10 │
    └───────────┘               └───────────┘
     任意实数                    Σ pᵢ = 1.00
                                pᵢ ∈ (0, 1)
```

**一句话**：Softmax = 指数放大差距 → 归一化成概率分布。

In [ ]:
# Softmax 的基础实现（含防溢出）

def softmax(z):
    """稳定的 Softmax 实现：减去最大值防止 e^z 溢出"""
    c = np.max(z, axis=-1, keepdims=True)  # 平移不变性
    exp_z = np.exp(z - c)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# 演示
z = np.array([2.0, 1.0, 0.1])
p = softmax(z)

print('输入 (logits):', z)
print('输出 (概率):  ', np.round(p, 4))
print(f'概率和 = {p.sum():.10f}')
print(f'最大对应: z[{z.argmax()}] = {z.max()} → p[{p.argmax()}] = {p.max():.4f}')

### 6.2 性质一：平移不变性 ⭐ 最重要

$$\text{softmax}(z_i + c) = \text{softmax}(z_i) \quad \text{对任意常数 } c \text{ 成立}$$

```
证明 (一行):

                e^{z_i + c}          e^{z_i} · e^c          e^{z_i}
softmax(z_i+c) = ────────────  =  ────────────────  =  ────────────  = softmax(z_i)
                 K                    K                     K
                 Σ e^{z_k + c}        Σ e^{z_k} · e^c       Σ e^{z_k}
                k=1                  k=1                   k=1

                                    e^c 约掉!
```

**实践意义**：
- 🛡️ 减去 max(z) 可防止 `e^z` 上溢（z=1000 → e¹⁰⁰⁰ = ∞ 在 float64 中）
- 🎯 不改变输出概率分布
- 💡 这就是为什么所有库的 Softmax 都写 `exp(z - max(z))`

In [ ]:
# 验证平移不变性
z = np.array([2.0, 1.0, 0.1])

p_original = softmax(z)
p_shifted_5  = softmax(z + 5)
p_shifted_n5 = softmax(z - 5)

print('原始 z:      ', z)
print('softmax(z):  ', np.round(p_original, 6))
print()
print('z + 5:       ', z + 5)
print('softmax(z+5):', np.round(p_shifted_5, 6))
print()
print('z - 5:       ', z - 5)
print('softmax(z-5):', np.round(p_shifted_n5, 6))
print()
print(f'三组结果完全一致: {np.allclose(p_original, p_shifted_5) and np.allclose(p_original, p_shifted_n5)}')
print()

# 演示数值稳定性: 没有减 max 会怎样？
z_large = np.array([1000.0, 100.0, 10.0])
print(f'z = {z_large}')
print(f'e^1000 = {np.exp(1000.0)}  ← 直接算会溢出!')
print(f'减去 max 后: softmax(z) = {softmax(z_large)} ← 正常工作')

### 6.3 性质二：保序性（单调递增）

$$z_i > z_j \quad \Longrightarrow \quad \text{softmax}(z_i) > \text{softmax}(z_j)$$

```
   Softmax 不改变输入的相对大小!

   z:   [3.0,    1.0,    0.5]     从大到小
        ↓       ↓       ↓
   p:   [0.84,   0.11,   0.04]    同样的顺序
```

**推论**：
- 推理时可以直接 `argmax(z)` 而**不需要算 softmax**（结果一样，更快）
- 分类问题的损失函数只看概率，不看 logit 绝对大小
- 这意味着 softmax **只是「翻译」，不是「篡改」**

In [ ]:
# 验证保序性
z = np.array([5.0, 2.0, -1.0, 3.0, 0.0])
p = softmax(z)

print('输入 z :', z)
print('输出 p :', np.round(p, 4))
print()
print('z 的排序 :', np.argsort(z)[::-1])
print('p 的排序 :', np.argsort(p)[::-1])
print(f'排序一致: {np.array_equal(np.argsort(z), np.argsort(p))}')
print(f'argmax 一致: z[{z.argmax()}] vs p[{p.argmax()}] → 推理时可省略 softmax')

### 6.4 性质三：非负且归一化（天然的概率解释）

$$0 < \text{softmax}(z_i) < 1 \quad \text{且} \quad \sum_{i=1}^{K}\text{softmax}(z_i) = 1$$

这是 softmax 被选为多分类输出的核心原因：**直接当概率用**。

```
 为什么不用 z 直接除以 sum(z)?

  z = [2, -1, 0.5]  →  sum(z) = 1.5
  z / sum(z) = [1.33, -0.67, 0.33]   ← 有负数! 不是概率!

  指数化后:  e^z = [7.39, 0.37, 1.65]  全部正数
  再归一化:  p = [0.79, 0.04, 0.18]    ✓ 合法概率分布
```

### 6.5 性质四：温度参数 τ — 控制分布的"软硬"

$$\text{softmax}(z_i, \tau) = \frac{e^{z_i / \tau}}{\sum_{k} e^{z_k / \tau}}$$

```
                    τ → 0+                     τ = 1                  τ → ∞
                    ─────────                  ─────                  ─────────
                    ┌──┐                                                ┌──┐
                    │  │                                                │  │
                    │  │  ← one-hot                              平均 →│  │
                    │  │    只选最大的                          均匀 →│  │
                    │  │                                                │  │
                    └──┘                                                └──┘
                    hard                     soft                      uniform
                   (argmax)                                        (所有类一样)

实际应用:
  τ < 1:  让分布更"尖锐"  →  知识蒸馏中教师网络的"硬标签"
  τ = 1:  标准 softmax
  τ > 1:  让分布更"平滑"  →  知识蒸馏中学生网络学习的"软标签"
```

In [ ]:
# 温度参数演示
z = np.array([3.0, 1.0, 0.5])
taus = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, tau in enumerate(taus):
    p = softmax(z / tau)
    bars = axes[i].bar(range(len(p)), p, color=['#E74C3C', '#3498DB', '#2ECC71'], edgecolor='white')
    axes[i].set_xticks(range(len(p)))
    axes[i].set_xticklabels([f'z={z[0]}', f'z={z[1]}', f'z={z[2]}'])
    axes[i].set_ylim(0, 1.05)
    axes[i].set_title(f'τ = {tau}')
    axes[i].set_ylabel('概率')
    for bar, val in zip(bars, p):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.3f}', ha='center', fontsize=10)

fig.suptitle('Softmax 温度参数的影响: τ 越小越 hard，τ 越大越 uniform', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 6.6 性质五：与交叉熵的完美配合

Softmax + CrossEntropy 组合在一起，反向传播的梯度极其简洁：

$$\frac{\partial L}{\partial z_i} = p_i - y_i \quad \text{（预测 - 标签）}$$

```
  为什么这很重要?

  ┌─────────────────────────────────────────────────────────┐
  │                                                         │
  │  分开写:  Softmax → CrossEntropyLoss → 反向传播         │
  │           需要算 softmax 的雅可比矩阵 (K×K)              │
  │           + 交叉熵的雅可比 (K×1)                        │
  │           + 链式法则相乘                                 │
  │                                                         │
  │  合起来:  ∂L/∂z = p - y                                │
  │           3 个字符搞定!                                  │
  │           不需要中间变量                                 │
  │           不需要额外内存                                 │
  │           数值更稳定（避免了除法/指数的梯度爆炸）          │
  │                                                         │
  └─────────────────────────────────────────────────────────┘
```

**所以 PyTorch 里都这么写**：
```python
# ❌ 不要这样（数值不稳定 + 低效）
loss = nn.NLLLoss()(torch.log(F.softmax(logits)), target)

# ✅ 正确做法
loss = nn.CrossEntropyLoss()(logits, target)  # 内部融合了 softmax
```

> `CrossEntropyLoss` 接受的是**原始 logits**，不是 softmax 后的概率。

### 6.7 性质六：多对一映射（不是双射）

$$\text{softmax}(z) = \text{softmax}(z + c \cdot [1, 1, \dots, 1])$$

平移不变性意味着：**无穷多组不同的 logits 映射到同一个概率分布**。

```
  z = [2, 1, 0.1]    →  p = [0.66, 0.24, 0.10]
  z = [5, 4, 3.1]    →  p = [0.66, 0.24, 0.10]   ← 一样的!
  z = [-3, -4, -5]   →  p = [0.66, 0.24, 0.10]   ← 还是一样的!

  差异被 e^c 因子吸收并约掉了
```

**这对优化意味着什么？**
- 交叉熵损失对这个偏移**不敏感**（因为 loss 只看概率）
- 但权重可以无限制增长 → 需要 **L2 正则化**来约束
- 实践中通常不加 bias 到最后一层（或用 weight decay 控制）

In [ ]:
# 可视化: softmax 如何把 2D logits 映射到概率单纯形
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 在 2D logit 空间采样
zs_2d = np.random.uniform(-3, 3, (500, 2))
colors = np.array(['C3' if z[0] > z[1] else 'C0' for z in zs_2d])
axes[0].scatter(zs_2d[:, 0], zs_2d[:, 1], c=colors, alpha=0.5, s=30)
axes[0].axhline(0, color='gray', alpha=0.2)
axes[0].axvline(0, color='gray', alpha=0.2)
axes[0].plot([-3, 3], [-3, 3], 'k--', alpha=0.3, label='z₁ = z₂')
axes[0].set_xlabel('z₁'); axes[0].set_ylabel('z₂')
axes[0].set_title('Logit 空间 (R²)')
axes[0].legend()
axes[0].set_aspect('equal')

# 右图: softmax 后映射到概率单纯形
ps = softmax(zs_2d)
axes[1].scatter(ps[:, 0], ps[:, 1], c=colors, alpha=0.5, s=30)
axes[1].plot([0, 1], [1, 0], 'k--', alpha=0.3, label='p₁ + p₂ = 1')
axes[1].axhline(0.5, color='gray', alpha=0.2)
axes[1].axvline(0.5, color='gray', alpha=0.2)
axes[1].set_xlabel('p₁'); axes[1].set_ylabel('p₂')
axes[1].set_title('Softmax 映射后 → 概率单纯形')
axes[1].legend()
axes[1].set_xlim(-0.05, 1.05); axes[1].set_ylim(-0.05, 1.05)
axes[1].set_aspect('equal')

plt.suptitle('Softmax 的几何: 把 R² 压缩到一条线段上 (p₁+p₂=1)', fontsize=13)
plt.tight_layout()
plt.show()

### 6.8 Softmax 六大性质总结

```
┌────────────────────────────────────────────────────────────────┐
│                       Softmax 性质全景                         │
├─────┬──────────────────────┬───────────────────────────────────┤
│  #  │ 性质                 │ 实际意义                          │
├─────┼──────────────────────┼───────────────────────────────────┤
│  1  │ 平移不变性           │ 减 max 防溢出, 所有框架标配       │
│  2  │ 保序性 (单调递增)    │ 推理时可省略, 直接用 argmax       │
│  3  │ 非负 + 归一化        │ 天然的概率分布, 可直接解释         │
│  4  │ 温度参数 τ           │ 控制软硬: 知识蒸馏, 采样多样性     │
│  5  │ CE 梯度 = p - y      │ 反向传播极简, 数值稳定, 省内存     │
│  6  │ 多对一映射           │ 需正则化约束权重, 不加 bias        │
└─────┴──────────────────────┴───────────────────────────────────┘
```

## 7. 激活函数对比速查表

| 激活函数 | 公式 | 值域 | 零中心 | 可微 | 饱和 | 使用场景 |
|----------|------|------|--------|------|------|----------|
| **阶跃** | 1 if x>0 else 0 | {0,1} | ✗ | ✗ | ✗ | 感知机（仅用于理解概念）|
| **Sigmoid** | 1/(1+e⁻ˣ) | (0,1) | ✗ | ✓ | ✓ | 二分类输出层、门控 |
| **Tanh** | (eˣ-e⁻ˣ)/(eˣ+e⁻ˣ) | (-1,1) | ✓ | ✓ | ✓ | RNN/LSTM 内部 |
| **ReLU** | max(0,x) | [0,∞) | ✗ | ✓ | ✗ | CNN/MLP 隐藏层（默认首选）|
| **Leaky ReLU** | max(0.01x, x) | (-∞,∞) | ✗ | ✓ | ✗ | 替代 ReLU，防 dead neuron |
| **GELU** | x·Φ(x) | (-∞,∞) | ✗ | ✓ | ✗ | Transformer (BERT, GPT) |
| **Softmax** | eᶻⁱ/Σeᶻᵏ | (0,1) | — | ✓ | ✗ | 多分类输出层 |

### 选择指南

```
  隐藏层:   ReLU → 不好使就换 GELU（Transformer）或 Leaky ReLU
  二分类输出: Sigmoid
  多分类输出: Softmax（配合 CrossEntropyLoss）
  回归输出:   恒等函数（不用激活）
  RNN:        Tanh（传统）/ ReLU（现代 GRU）
```

## 8. 小结

### 三个核心认知

```
1. 激活函数的存在意义 = 引入非线性
   └→ 没有它, 1000 层 = 1 层

2. 隐藏层用 ReLU 基本没错
   └→ 简单、快、不饱和。Dead neuron 用 Leaky/PReLU/GELU 治

3. Softmax 是输出层的专属
   └→ 六大性质都记住: 平移不变、保序、归一化、温度、CE 梯度、多对一
```

### 一句话

> **激活函数 = 神经网络的"心跳"**。没有非线性，就只是一堆矩阵乘法的堆叠。ReLU 负责隐层传播，Softmax 负责最后裁决。